In [9]:
import scanpy as sc
import harmonypy as hm
import tqdm as notebook_tqdm
import phate
import pandas as pd
from matplotlib.colors import to_rgba
from pygam import LinearGAM, s
import meld
import scipy.stats as stats
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import combinations
import gseapy as gp
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator
import scprep

In [10]:
adata= sc.read_h5ad("/Users/Jan/data/final_final_data.h5ad")

In [ ]:
# Figure 4A
sc.external.pl.phate(adata, color="predicted_labels", title= "Predicted Cell Type", frameon= False)

In [ ]:
# Figures 4C- I
adata= adata[adata.obs["predicted_labels"]!= "Outlier 1 (Stressed)"]
ct = pd.crosstab(
    adata.obs["Sample name"],
    adata.obs["predicted_labels"],
    normalize="index"
)
comparisons= {
    "NoTx": ("1-WT-No-Tx", "1-KA-No-Tx"),
    "WT-1": ("1-WT-No-Tx", "1-WT-1h"),
    "WT-3": ("1-WT-1h", "1-WT-4h"),
    "WT-2": ("1-WT-No-Tx", "1-WT-4h"),
    "KA-1": ("1-KA-No-Tx", "1-KA-1h"),
    "KA-3": ("1-KA-1h", "1-KA-4h"),
    "KA-2": ("1-KA-No-Tx", "1-KA-4h"),
    "4h": ("1-WT-4h", "1-KA-4h")
}
logfc= {}
for name, (t1, t2) in comparisons.items():
    fc= np.log2((ct.loc[t2]+ 1e-6)/ (ct.loc[t1]+ 1e-6))
    logfc[name]= fc
logfc_df= pd.DataFrame(logfc)
xmin= logfc_df.min().min()
xmax= logfc_df.max().max()
lim= max(abs(xmin), abs(xmax))
lim+= 0.1* lim
for col in logfc_df.columns:
    fig, ax= plt.subplots(figsize= (6, 4))
    logfc_df[col].sort_values().plot(kind= "barh", ax= ax)
    ax.set_xlim(-lim, lim)
    ax.axvline(0, color= "black", linewidth= 1)
    ax.set_title(f"{comparisons[col]}", fontsize= 11)
    ax.set_xlabel("Log fold change")
    ax.set_ylabel("Predicted cell types")
    fig.tight_layout()
    plt.show()

In [ ]:
# Figure 4J
adata_untreated= adata[adata.obs['Timepoint']== 'No-Tx']
meld_scores_list= []
cell_types= ['TA 2', 'TA 1', 'proCSC', 'CSC', 'revCSC', 'Early Enterocyte', 'Late Enterocyte', 'ER Stress', 'Goblet or DCS']
for cell_type in cell_types:      
    adata_filtered_by_cell_type= adata_untreated[adata_untreated.obs["predicted_labels"]== cell_type]
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(adata_filtered_by_cell_type, sample_labels=adata_filtered_by_cell_type.obs['Sample Type'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    cols = sorted(adata_filtered_by_cell_type.obs['Sample Type'].unique())
    densities_df= pd.DataFrame(
        sample_densities,
        index= adata_filtered_by_cell_type.obs_names,
        columns= cols
    )
    ka_col = "KA"
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df["KA"],
        "Comparison": cell_type
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_map = {c: str(i) for i, c in enumerate(cell_types)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
meld_scores = meld_scores.sort_values("Comparison")
fig, ax = plt.subplots(figsize=(10, 10))
ax= scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=90)
plt.xlabel("Cell Type")
plt.title("MELD Likelihoods of Different Cell Types KA vs WT Untreated")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4M part 1
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)

In [ ]:
# Figure 4K
adata_wt= adata[adata.obs["Sample Type"]== "WT"]
meld_scores_list= []
cell_types= ['TA 2', 'TA 1', 'proCSC', 'CSC', 'revCSC', 'Early Enterocyte', 'Late Enterocyte', 'ER Stress', 'Goblet or DCS']
adata_wt_no_1h= adata_wt[adata_wt.obs["Timepoint"]!= "1h"]
for cell_type in cell_types:      
    adata_filtered_by_cell_type= adata_wt_no_1h[adata_wt_no_1h.obs["predicted_labels"]== cell_type]
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(adata_filtered_by_cell_type, sample_labels=adata_filtered_by_cell_type.obs['Timepoint'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    cols = sorted(adata_filtered_by_cell_type.obs['Timepoint'].unique())
    densities_df= pd.DataFrame(
        sample_densities,
        index= adata_filtered_by_cell_type.obs_names,
        columns= cols
    )
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df["No-Tx"],
        "Comparison": cell_type
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_map = {c: str(i) for i, c in enumerate(cell_types)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
fig, ax = plt.subplots(figsize=(10, 10))
ax= scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=90)
plt.xlabel("Cell Type")
plt.title("MELD Likelihoods of Different Cell Types WT 4h vs WT Untreated")
plt.tight_layout()
plt.savefig("C:/Users/Jan/data/chris outputs/integrated_final/more_graphs/wt/meld jitter", bbox_inches= "tight")
plt.show()

In [ ]:
# Figure 4M part 2
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)

In [ ]:
# Figure 4L
adata_ka= adata[adata.obs["Sample Type"]== "KA"]
meld_scores_list= []
cell_types= ['TA 2', 'TA 1', 'proCSC', 'CSC', 'revCSC', 'Early Enterocyte', 'Late Enterocyte', 'ER Stress', 'Goblet or DCS']
adata_ka_no_1h= adata_ka[adata_ka.obs["Timepoint"]!= "1h"]
for cell_type in cell_types:      
    adata_filtered_by_cell_type= adata_ka_no_1h[adata_ka_no_1h.obs["predicted_labels"]== cell_type]
    meld_op = meld.MELD(random_state= 12)
    sample_densities= meld_op.fit_transform(adata_filtered_by_cell_type, sample_labels=adata_filtered_by_cell_type.obs['Timepoint'])
    sample_densities= sk.preprocessing.normalize(
        sample_densities,
        norm='l1',
        axis=1
    )
    cols = sorted(adata_filtered_by_cell_type.obs['Timepoint'].unique())
    densities_df= pd.DataFrame(
        sample_densities,
        index= adata_filtered_by_cell_type.obs_names,
        columns= cols
    )
    meld_score = pd.DataFrame({
        "MELD Likelihood": densities_df["No-Tx"],
        "Comparison": cell_type
    })
    meld_scores_list.append(meld_score)
meld_scores= pd.concat(meld_scores_list)
comparison_map = {c: str(i) for i, c in enumerate(cell_types)}
meld_scores["comparison_sort"] = meld_scores["Comparison"].map(comparison_map)
meld_scores["comparison_sort"] = meld_scores["comparison_sort"].astype('category')
fig, ax = plt.subplots(figsize=(10, 10))
ax= scprep.plot.jitter(meld_scores["comparison_sort"], meld_scores["MELD Likelihood"], 
                    ax= ax,
                   plot_means=False)
ax.set_ylim(0, 1)
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=90)
plt.xlabel("Cell Type")
plt.title("MELD Likelihoods of Different Cell Types KA 4h vs KA Untreated")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4M part 3
meld_scores["Polarisation"]= (meld_scores["MELD Likelihood"]- 0.5).abs()
summary_stats= meld_scores.groupby("Comparison")["Polarisation"].mean().reset_index()
print(summary_stats)